# Lesson 16: Capstone, multimodal Forkcast

Goes with the Lesson 16 page of the Forkcast course. One model for a listing's photo, description and numbers; an ablation over modalities; a model card; the artifact; and the whole test suite green.

Runtime: CPU is enough (about half a minute for the full ablation).

## 1. Clone your repository and install the package

In [ ]:
GITHUB_USER = "your-github-username"   # edit this line
REPO = "forkcast-course"

!git clone https://github.com/{GITHUB_USER}/{REPO}.git
%cd {REPO}
!pip install -e .

## 2. `forkcast/capstone.py`

The data generator, dataset and loaders are provided. Write `collate_listings`, `FusionModel`, `run_ablation` and `write_model_card` as the page builds them.

In [ ]:
%%writefile forkcast/capstone.py
"""Lesson 16, the capstone. One model for a Forkcast listing: its photo, its description and its numbers, fused.

``make_listings``, ``ListingsDataset``, ``listing_loaders`` and ``accuracy_fusion`` are provided; the rest are exercises.
"""

from __future__ import annotations

import random
from pathlib import Path

import torch
from torch import Tensor, nn
from torch.utils.data import DataLoader, Dataset

from forkcast.model import MLP
from forkcast.text import Vocab, pad_batch, tokenize
from forkcast.vision import SHAPE_CLASSES, make_shapes

MODALITIES = ("image", "text", "tabular")
TAB_FEATURES = ["price", "prep_minutes", "rating"]
DISH_PULL = [1.6, -0.5, 0.9, -1.5]            # how much each photo category helps demand (circle, square, triangle, cross)
PORTION = {"generous": 1.0, "decent": 0.3, "small": -0.8, "tiny": -1.2}
FRESH = {"fresh": 0.6, "homemade": 0.5, "standard": 0.0, "frozen": -0.7, "reheated": -0.9}


def make_listings(n: int = 1600, seed: int = 0, size: int = 32):
    """Synthetic listings. Returns (images (n, 3, size, size), texts: list[str], tabular (n, 3), labels (n,) int64).

    Demand depends on all three: the photo category (DISH_PULL), the words in the description (PORTION, FRESH),
    and price, prep time and rating. The label is 1 when demand is above the median: a balanced "will it sell" task
    that no single modality can solve on its own.
    """
    images, categories = make_shapes(n, seed=seed, size=size)
    rng = random.Random(seed)
    g = torch.Generator().manual_seed(seed + 1)
    texts, text_score = [], []
    for _ in range(n):
        p, f = rng.choice(list(PORTION)), rng.choice(list(FRESH))
        name = rng.choice(["pad thai", "ramen", "pizza", "pho", "lasagna", "tacos", "curry", "burger"])
        texts.append(rng.choice([f"{p.capitalize()} portion of {f} {name}.", f"{f.capitalize()} {name}, {p} portion.", f"Our {name}: {f}, {p} serving."]))
        text_score.append(PORTION[p] + FRESH[f])
    price = 6 + 24 * torch.rand(n, generator=g)
    prep = 10 + 40 * torch.rand(n, generator=g)
    rating = 3.0 + 2.0 * torch.rand(n, generator=g)
    tabular = torch.stack([price, prep, rating], dim=1)
    demand = (torch.tensor(DISH_PULL)[categories] + 1.0 * torch.tensor(text_score) - 0.08 * price + 0.9 * rating - 0.02 * prep
              + 0.4 * torch.randn(n, generator=g))
    labels = (demand > demand.median()).long()
    return images, texts, tabular, labels


class ListingsDataset(Dataset):
    def __init__(self, images: Tensor, texts: list[str], tabular: Tensor, labels: Tensor, vocab: Vocab, tab_mean: Tensor, tab_std: Tensor):
        self.images, self.labels = images, labels
        self.ids = [torch.tensor(vocab.encode(tokenize(t)), dtype=torch.long) for t in texts]
        self.tabular = (tabular - tab_mean) / tab_std

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, i: int):
        return self.images[i], self.ids[i], self.tabular[i], self.labels[i]


def collate_listings(batch):
    """A list of (image, ids, tabular, label) -> (images (N, 3, H, W), ids (N, T), mask (N, T), tabular (N, 3), labels (N,)).

    Stack what stacks, pad the text with pad_batch, and keep the label last for the toolkit.
    """
    raise NotImplementedError("Lesson 16, step 2: images, ids, tabular, labels = zip(*batch); pad_batch(list(ids)); torch.stack the rest")


class FusionModel(nn.Module):
    """Three encoders, one head. Each encoder maps its modality to a d-vector; the three are concatenated and classified.

    image: SmallCNN's two conv blocks (3 -> 16 -> 32 channels, two pools) then Flatten and Linear(32 * (size//4)^2, d).
    text: nn.Embedding(vocab, d, padding_idx=0) with Lesson 11's masked mean.
    tabular: MLP(3, (d,)) from Lesson 4 but with d outputs: use nn.Sequential(nn.Linear(3, d), nn.ReLU(), nn.Linear(d, d)).
    head: nn.Sequential(nn.Linear(3 * d, hidden), nn.ReLU(), nn.Linear(hidden, 2)).
    ``modalities`` names the inputs that are used; an unused modality contributes a zero vector, so the same
    class runs every ablation.
    """

    def __init__(self, vocab_size: int, d: int = 32, hidden: int = 64, size: int = 32, modalities=MODALITIES):
        super().__init__()
        self.modalities = tuple(modalities)
        self.d = d
        raise NotImplementedError("Lesson 16, step 3: self.image_encoder, self.image_proj, self.text_embedding, self.tab_encoder, self.head")

    def encode_image(self, images: Tensor) -> Tensor:
        raise NotImplementedError("Lesson 16, step 3: conv blocks -> flatten -> Linear to d")

    def encode_text(self, ids: Tensor, mask: Tensor) -> Tensor:
        raise NotImplementedError("Lesson 16, step 3: embedding, masked mean (Lesson 11)")

    def forward(self, images: Tensor, ids: Tensor, mask: Tensor, tabular: Tensor) -> Tensor:
        """Concatenate the used encoders' outputs (zeros for the unused ones) and classify. -> (N, 2)"""
        raise NotImplementedError("Lesson 16, step 3: encode each used modality, zeros of shape (N, d) for unused ones, torch.cat(dim=1), head")


def listing_loaders(n_train: int = 1200, n_val: int = 400, batch_size: int = 64):
    """Train and validation loaders plus the vocabulary and the tabular statistics (from the training part only)."""
    images, texts, tabular, labels = make_listings(n_train + n_val, seed=0)
    vocab = Vocab([tokenize(t) for t in texts[:n_train]])
    tab_mean, tab_std = tabular[:n_train].mean(dim=0), tabular[:n_train].std(dim=0, correction=0)
    train_ds = ListingsDataset(images[:n_train], texts[:n_train], tabular[:n_train], labels[:n_train], vocab, tab_mean, tab_std)
    val_ds = ListingsDataset(images[n_train:], texts[n_train:], tabular[n_train:], labels[n_train:], vocab, tab_mean, tab_std)
    return (DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=collate_listings),
            DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_listings), vocab, (tab_mean, tab_std))


@torch.no_grad()
def accuracy_fusion(model: nn.Module, loader: DataLoader, device="cpu") -> float:
    model.eval(); correct = total = 0
    for *inputs, y in loader:
        inputs = [x.to(device) for x in inputs]
        correct += (model(*inputs).argmax(dim=1).cpu() == y).sum().item(); total += len(y)
    return correct / total


def run_ablation(make_model, train_loader: DataLoader, val_loader: DataLoader, combos, epochs: int = 8, lr: float = 1e-3, device="cpu") -> dict:
    """For each tuple of modalities in ``combos``: build make_model(modalities), train it with the toolkit's
    fit_v2 (CrossEntropyLoss), and record validation accuracy. Returns {combo: accuracy}. Seed with torch.manual_seed(0)
    before building each model so the comparison is fair."""
    raise NotImplementedError("Lesson 16, step 4: loop over combos, torch.manual_seed(0), train.fit_v2(..., loss_fn=nn.CrossEntropyLoss()), accuracy_fusion")


def write_model_card(path: str | Path, name: str, intended_use: str, data: str, metrics: dict, limitations: list[str], lesson_trail: list[str]) -> str:
    """A Markdown model card with the sections every deployed model should ship with. Returns the text and writes it.

    Sections, as level-2 headings in this order: Intended use, Training data, Evaluation, Limitations, Provenance.
    Evaluation renders ``metrics`` as a two-column table (metric | value, values formatted with 3 decimals when numeric).
    Limitations renders one bullet per entry; Provenance lists the lesson trail (which lesson built each part) as bullets.
    """
    raise NotImplementedError("Lesson 16, step 6: build the Markdown string section by section, write it, return it")

## 3. The listings

In [ ]:
import importlib, itertools, torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torchvision.utils import make_grid
from forkcast import capstone as cap, train, debug
from forkcast.model import count_parameters
importlib.reload(cap)

images, texts, tabular, labels = cap.make_listings(1600)
print(images.shape, tabular.shape, "label balance", labels.float().mean().item())
plt.figure(figsize=(12, 2.4)); plt.axis("off"); plt.imshow(make_grid(images[:8], nrow=8).permute(1, 2, 0))
for t, tab, l in zip(texts[:8], tabular[:8], labels[:8]):
    print(f"{l.item()}  {t:45s} price {tab[0]:5.1f}  prep {tab[1]:4.0f}  rating {tab[2]:.1f}")
train_loader, val_loader, vocab, (tab_mean, tab_std) = cap.listing_loaders()
images_b, ids, mask, tabular_b, labels_b = next(iter(train_loader))
print(images_b.shape, ids.shape, mask.shape, tabular_b.shape, labels_b.shape)

## 4. The fusion model

In [ ]:
importlib.reload(cap)
model = cap.FusionModel(len(vocab))
print(model(images_b, ids, mask, tabular_b).shape, "|", count_parameters(model), "parameters")
tab_only = cap.FusionModel(len(vocab), modalities=("tabular",))
print("tabular-only ignores the photo:", torch.allclose(tab_only(images_b, ids, mask, tabular_b), tab_only(torch.rand_like(images_b), ids, mask, tabular_b)))

## 5. The ablation: all seven combinations

In [ ]:
importlib.reload(cap)
combos = [c for r in range(1, 4) for c in itertools.combinations(cap.MODALITIES, r)]
results = cap.run_ablation(lambda m: cap.FusionModel(len(vocab), modalities=m), train_loader, val_loader, combos, epochs=12, lr=2e-3)
for k, v in results.items():
    print(f"{' + '.join(k):26s} {v:.3f}")
plt.figure(figsize=(8, 3)); plt.bar([" + ".join(k) for k in results], list(results.values())); plt.xticks(rotation=30, ha="right"); plt.axhline(0.5, ls="--", c="k"); plt.ylabel("validation accuracy");

## 6. Train the fused model and evaluate it the Lesson 10 way

In [ ]:
torch.manual_seed(0)
fused = cap.FusionModel(len(vocab))
h = train.fit_v2(fused, train_loader, val_loader, epochs=12, lr=2e-3, loss_fn=nn.CrossEntropyLoss())
print("fused validation accuracy:", round(cap.accuracy_fusion(fused, val_loader), 3))
fused.eval(); logits_all, y_all, imgs_all = [], [], []
with torch.no_grad():
    for im, i_, mk, tb, y in val_loader:
        logits_all.append(fused(im, i_, mk, tb)); y_all.append(y); imgs_all.append(im)
logits_all, y_all, imgs_all = torch.cat(logits_all), torch.cat(y_all), torch.cat(imgs_all)
probs = torch.softmax(logits_all, 1)
print("ECE:", round(debug.expected_calibration_error(probs, y_all), 3))
picks = debug.misclassified(logits_all, y_all, k=8)
print("most confident mistakes:", [(i, t, p, round(c, 2)) for i, t, p, c in picks[:4]])
plt.figure(figsize=(12, 2.4)); plt.axis("off"); plt.imshow(debug.gallery(imgs_all, picks).permute(1, 2, 0))

## 7. The model card

In [ ]:
importlib.reload(cap)
card = cap.write_model_card("MODEL_CARD.md", "Forkcast listing demand v1",
    intended_use="Rank new listings for review priority. Not for pricing decisions.",
    data="1,600 synthetic listings: Lesson 7 shapes as photos, templated descriptions, simulated prices, prep times and ratings.",
    metrics={"accuracy (fused)": results[("image", "text", "tabular")], "accuracy (best single modality)": max(v for k, v in results.items() if len(k) == 1), "epochs": 12},
    limitations=["Synthetic data only; real photos and reviews will shift the distribution.", "32x32 images; a real dish model would use Lesson 9's pretrained backbone.", "The label is 'above median demand', not observed orders."],
    lesson_trail=["Lesson 7: image encoder (two conv blocks)", "Lesson 11: text encoder (embedding, masked mean)", "Lesson 4: tabular encoder (MLP)", "Lesson 8: training loop v2", "Lesson 15: artifact format"])
print(card)

## 8. Ship it as an artifact

In [ ]:
import os; os.makedirs("artifacts", exist_ok=True)
torch.save({"state_dict": fused.state_dict(), "itos": vocab.itos, "tab_mean": tab_mean, "tab_std": tab_std, "d": 32, "hidden": 64, "size": 32, "modalities": list(fused.modalities)}, "artifacts/listing.pt")
bundle = torch.load("artifacts/listing.pt", weights_only=True)
fresh = cap.FusionModel(len(bundle["itos"]), d=bundle["d"], hidden=bundle["hidden"], size=bundle["size"], modalities=tuple(bundle["modalities"]))
fresh.load_state_dict(bundle["state_dict"]); fresh.eval()
with torch.inference_mode():
    print("identical predictions after reload:", torch.equal(fresh(images_b, ids, mask, tabular_b), fused(images_b, ids, mask, tabular_b)))

### Break it on purpose: raw tabular inputs

In [ ]:
raw_mean, raw_std = torch.zeros(3), torch.ones(3)        # no standardization
imgs_, texts_, tab_, labels_ = cap.make_listings(1600)
raw_train = torch.utils.data.DataLoader(cap.ListingsDataset(imgs_[:1200], texts_[:1200], tab_[:1200], labels_[:1200], vocab, raw_mean, raw_std), batch_size=64, shuffle=True, collate_fn=cap.collate_listings)
raw_val = torch.utils.data.DataLoader(cap.ListingsDataset(imgs_[1200:], texts_[1200:], tab_[1200:], labels_[1200:], vocab, raw_mean, raw_std), batch_size=64, collate_fn=cap.collate_listings)
torch.manual_seed(0); raw_model = cap.FusionModel(len(vocab))
train.fit_v2(raw_model, raw_train, raw_val, epochs=12, lr=2e-3, loss_fn=nn.CrossEntropyLoss())
print("fused, raw tabular inputs:", round(cap.accuracy_fusion(raw_model, raw_val), 3), "| standardized:", round(results[("image", "text", "tabular")], 3))

## 9. Definition of done

Expected: `7 passed`, then `118 passed` for the whole course (the only skips are the GPU test and the optional Lesson 17).

In [ ]:
!pytest tests/test_16.py -q

In [ ]:
!pytest -m fast -q

## 10. Commit and push

In [ ]:
from getpass import getpass
TOKEN = getpass("GitHub token with access to your repo: ")

!git config user.name "{GITHUB_USER}"
!git config user.email "{GITHUB_USER}@users.noreply.github.com"
!git add -A
!git commit -m "Lesson 16: capstone, multimodal Forkcast"
!git push https://{TOKEN}@github.com/{GITHUB_USER}/{REPO}.git HEAD:main